In [2]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression,SGDRegressor

from sklearn.preprocessing import PolynomialFeatures,StandardScaler

from sklearn.metrics import r2_score,accuracy_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,root_mean_squared_error
from sklearn.pipeline import Pipeline
import pandas as pd

In [3]:
df=pd.read_csv('1553768847-housing.csv')

In [4]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-122.23,37.88,41,880,129.0,322,126,8.3252,NEAR BAY,452600
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,NEAR BAY,358500
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,NEAR BAY,352100
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,NEAR BAY,341300
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,NEAR BAY,342200


In [5]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [6]:
df.isnull().sum()
df['total_bedrooms'].fillna(df['total_bedrooms'].mean(), inplace=True)

C:\Users\nic\AppData\Local\Temp\ipykernel_17072\3206742078.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].mean(), inplace=True)


In [8]:
df.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
median_house_value    0
dtype: int64

In [9]:
df.columns=df.columns.str.capitalize()

In [10]:
df.head()

,Longitude,Latitude,Housing_median_age,Total_rooms,Total_bedrooms,Population,Households,Median_income,Ocean_proximity,Median_house_value
0,-122.23,37.88,41,880,129.0,322,126,8.3252,NEAR BAY,452600
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,NEAR BAY,358500
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,NEAR BAY,352100
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,NEAR BAY,341300
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,NEAR BAY,342200


In [11]:
df['Ocean_proximity'].unique()

array(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'],
      dtype=object)

In [12]:
from sklearn.preprocessing import OrdinalEncoder

# Define meaningful order (low → high value)
order = [['INLAND', '<1H OCEAN', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND']]

encoder = OrdinalEncoder(categories=order)

df['Ocean_proximity'] = encoder.fit_transform(df[['Ocean_proximity']])

In [13]:
df.head()

,Longitude,Latitude,Housing_median_age,Total_rooms,Total_bedrooms,Population,Households,Median_income,Ocean_proximity,Median_house_value
0,-122.23,37.88,41,880,129.0,322,126,8.3252,3.0,452600
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,3.0,358500
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,3.0,352100
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,3.0,341300
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,3.0,342200


In [14]:
df.corr()

,Longitude,Latitude,Housing_median_age,Total_rooms,Total_bedrooms,Population,Households,Median_income,Ocean_proximity,Median_house_value
Longitude,1.000000,-0.924664,-0.108197,0.044568,0.069260,0.099773,0.055310,-0.015176,-0.271730,-0.045967
Latitude,-0.924664,1.000000,0.011173,-0.036100,-0.066658,-0.108785,-0.071035,-0.079809,0.007695,-0.144160
Housing_median_age,-0.108197,0.011173,1.000000,-0.361262,-0.318998,-0.296244,-0.302916,-0.119034,0.295012,0.105623
Total_rooms,0.044568,-0.036100,-0.361262,1.000000,0.927253,0.857126,0.918484,0.198050,-0.031586,0.134153
Total_bedrooms,0.069260,-0.066658,-0.318998,0.927253,1.000000,0.873910,0.974725,-0.007682,-0.009970,0.049454
Population,0.099773,-0.108785,-0.296244,0.857126,0.873910,1.000000,0.907222,0.004834,-0.039415,-0.024650
Households,0.055310,-0.071035,-0.302916,0.918484,0.974725,0.907222,1.000000,0.013033,0.012873,0.065843
Median_income,-0.015176,-0.079809,-0.119034,0.198050,-0.007682,0.004834,0.013033,1.000000,0.163755,0.688075
Ocean_proximity,-0.271730,0.007695,0.295012,-0.031586,-0.009970,-0.039415,0.012873,0.163755,1.000000,0.397251
Median_house_value,-0.045967,-0.144160,0.105623,0.134153,0.049454,-0.024650,0.065843,0.688075,0.397251,1.000000


In [15]:
# Feature Engineering

df["rooms_per_household"] = df["Total_rooms"] / df["Households"]

df["bedrooms_per_room"] = df["Total_bedrooms"] / df["Total_rooms"]

df["population_per_household"] = df["Population"] / df["Households"]

In [16]:
df.head()

,Longitude,Latitude,Housing_median_age,Total_rooms,Total_bedrooms,Population,Households,Median_income,Ocean_proximity,Median_house_value,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41,880,129.0,322,126,8.3252,3.0,452600,6.984127,0.146591,2.555556
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,3.0,358500,6.238137,0.155797,2.109842
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,3.0,352100,8.288136,0.129516,2.802260
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,3.0,341300,5.817352,0.184458,2.547945
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,3.0,342200,6.281853,0.172096,2.181467


In [17]:
df.drop(['Total_rooms','Total_bedrooms','Population','Households'],axis=1,inplace=True)

In [18]:
df.corr()['Median_house_value']


Longitude                  -0.045967
Latitude                   -0.144160
Housing_median_age          0.105623
Median_income               0.688075
Ocean_proximity             0.397251
Median_house_value          1.000000
rooms_per_household         0.151948
bedrooms_per_room          -0.220049
population_per_household   -0.023737
Name: Median_house_value, dtype: float64

In [19]:
df.drop(['Longitude','population_per_household'],axis=1,inplace=True)

In [20]:
X=df.drop('Median_house_value',axis=1)
y=df['Median_house_value']

TypeError: scatter() missing 1 required positional argument: 'y'

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
X_train.shape



(15480, 6)

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=175.0)
model.fit(X_train, y_train)

Ridge(alpha=175.0)

In [ ]:
X_train.shape

(15480, 6)

In [ ]:
y_train.shape

(15480,)

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
mean_squared_error(y_test,y_pred)


5691574784.911888

In [ ]:
r2_score(y_test,y_pred)

0.5656642407508312

In [ ]:
root_mean_squared_error(y_test,y_pred)

75442.52636883184

In [ ]:
df["Median_house_value"].mean()

np.float64(206855.81690891474)

In [ ]:
''
''

''